In [102]:
import pandas as pd
import psycopg
from sqlalchemy import create_engine, inspect, DateTime
from datetime import datetime

In [103]:
smard = pd.read_csv("Gro_handelspreise_202301010000_202401010000_Stunde.csv", sep=";", na_values="-", decimal=",", thousands=".")
smardlog = smard[["Datum von", "Deutschland/Luxemburg [€/MWh] Originalauflösungen"]]
smardlog.rename(columns={"Datum von": "timestamp", "Deutschland/Luxemburg [€/MWh] Originalauflösungen": "price"}, inplace=True)

/tmp/ipykernel_67907/4130981027.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  smardlog.rename(columns={"Datum von": "timestamp", "Deutschland/Luxemburg [€/MWh] Originalauflösungen": "price"}, inplace=True)


In [104]:
dt = pd.read_csv("Datteln_202503010000_202504302359_Stunde.csv", sep=";", na_values="-", decimal=",", thousands=".")
dt.fillna(0.0, inplace=True)
dt["Datum von"] = pd.to_datetime(dt["Datum von"], format="mixed")

In [105]:
merged = smdl.merge(dt, left_on="timestamp", right_on="Datum von")

In [106]:
merged2 = merged[["timestamp", "Generation_DE Datteln 4 [MW] Originalauflösungen", "price"]]

In [107]:
merged2["revenue"] = merged2["Generation_DE Datteln 4 [MW] Originalauflösungen"] * merged2["price"]

/tmp/ipykernel_67907/4186250792.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged2["revenue"] = merged2["Generation_DE Datteln 4 [MW] Originalauflösungen"] * merged2["price"]


In [108]:
income = merged2["revenue"].sum() / len(merged2) # per hour

In [109]:
income

np.float64(23422.563333333335)

In [110]:
avg_power = dt["Generation_DE Datteln 4 [MW] Originalauflösungen"].sum() / len(dt)
avg_price = (smardlog["price"].sum() / len(smardlog)) or 120

In [111]:
revenue_per_hour = avg_power * avg_price

In [112]:
revenue_per_hour

np.float64(39790.23478796618)

In [113]:
(income - revenue_per_hour) / revenue_per_hour

np.float64(-0.41134895387907955)

In [114]:
revenue_per_hour + revenue_per_hour * ((income - revenue_per_hour) / revenue_per_hour)

np.float64(23422.563333333335)